# Investigation of `wwinp_1`

In [1]:
from kika.wwinp import read_wwinp
import numpy as np

FILEPATH = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/wwinp/wwinp_1"
ww = read_wwinp(FILEPATH)
print(ww)

WWINP  mesh_type=cartesian  particles=1  time_dep=no
  file: /mnt/c/Users/MONLEON-DE-LA-JAN/Documents/wwinp/wwinp_1
  Geometry:
    x: [-442, 440]  bins=182
    y: [-440, 445]  bins=216
    z: [-400, 400]  bins=120
  Particle 0:
    shape=(1, 5, 120, 216, 182)  energies=5
    min=7.516e-11  max=8.395e+19  nonzero=23587200/23587200 (100.0%)


## 1. Energy grid

In [2]:
e_bins = ww.energy_bins[0]
print(f"Number of energy bins: {len(e_bins)}")
print(f"\nUpper energy boundaries (MeV):")
for i, e in enumerate(e_bins):
    lo = 0.0 if i == 0 else e_bins[i - 1]
    print(f"  bin {i}: [{lo:.6e}, {e:.6e}] MeV")

Number of energy bins: 5

Upper energy boundaries (MeV):
  bin 0: [0.000000e+00, 1.422700e+00] MeV
  bin 1: [1.422700e+00, 1.826800e+00] MeV
  bin 2: [1.826800e+00, 3.011900e+00] MeV
  bin 3: [3.011900e+00, 6.376300e+00] MeV
  bin 4: [6.376300e+00, 2.000000e+01] MeV


## 2. Geometry

In [3]:
h = ww.header
fm = ww.geometry.fine_mesh
cm = ww.geometry.coarse_mesh
labels = ww.geometry.axis_labels

print(f"Mesh type: {h.mesh_type}")
print(f"Spatial shape: ({h.nfx}, {h.nfy}, {h.nfz})  = {h.n_spatial:,} cells")
print(f"Origin: ({h.x0}, {h.y0}, {h.z0})")
print(f"Coarse segments: ncx={h.ncx}, ncy={h.ncy}, ncz={h.ncz}")
print()

for label in labels:
    f_grid = fm[label]
    c_grid = cm[label]
    print(f"--- {label}-axis ---")
    print(f"  range: [{f_grid[0]:.2f}, {f_grid[-1]:.2f}]")
    print(f"  fine mesh:   {len(f_grid)-1} bins, {len(f_grid)} boundaries")
    print(f"  coarse mesh: {len(c_grid)-1} segments, boundaries = {c_grid}")
    # Show bin widths
    widths = np.diff(f_grid)
    print(f"  bin widths:  min={widths.min():.4f}, max={widths.max():.4f}, mean={widths.mean():.4f}")
    print()

Mesh type: cartesian
Spatial shape: (182, 216, 120)  = 4,717,440 cells
Origin: (-442.0, -440.0, -400.0)
Coarse segments: ncx=182, ncy=216, ncz=120

--- x-axis ---
  range: [-442.00, 440.00]
  fine mesh:   182 bins, 183 boundaries
  coarse mesh: 182 segments, boundaries = [-442.    -421.882 -401.765 -381.647 -361.529 -341.412 -321.294 -301.176
 -281.059 -260.941 -240.824 -220.706 -200.588 -180.471 -160.353 -140.235
 -120.118 -100.     -95.     -90.     -85.     -80.     -75.     -70.
  -65.     -60.     -55.     -50.     -45.     -40.     -35.     -30.
  -25.     -20.     -15.     -10.      -5.       0.       2.       4.
    6.       8.      10.      12.      14.      16.      18.      20.
   22.      24.      26.      28.      30.      32.      34.      36.
   38.      40.      42.      44.      46.      48.      50.      52.
   54.      56.      58.      60.      62.      64.      66.      68.
   70.      72.      74.      76.      78.      80.      82.      84.
   86.      88.      9

## 3. Value overview per energy bin

In [ ]:
from kika.wwinp.utils import calculate_ratios_stats

arr = ww.values.ww_values[0]  # (1, ne, nfz, nfy, nfx)

print(f"{'Bin':>4} {'E_lo (MeV)':>12} {'E_hi (MeV)':>12} {'Min':>12} {'Max':>12} {'Zero%':>7} {'Avg ratio':>10} {'Max ratio':>10}")
print("-" * 93)
for e in range(arr.shape[1]):
    spatial = arr[0, e]  # (nfz, nfy, nfx)
    e_lo = 0.0 if e == 0 else e_bins[e - 1]
    e_hi = e_bins[e]
    pos = spatial[spatial > 0]
    zero_pct = 100 * (1 - pos.size / spatial.size)
    avg_r, max_r = calculate_ratios_stats(spatial)
    print(f"{e:4d} {e_lo:12.4e} {e_hi:12.4e} {pos.min():12.4e} {pos.max():12.4e} {zero_pct:6.1f}% {avg_r:10.2f} {max_r:10.2f}")

## 4. Query specific cells

Change `ENERGY_IDX`, `X_VAL`, `Y_VAL`, `Z_VAL` below to inspect values at any location.

In [ ]:
# ── Pick a point ──────────────────────────────────────────────
X_VAL = 0.0    # x coordinate (cm)
Y_VAL = 0.0    # y coordinate (cm)
Z_VAL = 0.0    # z coordinate (cm)
# ──────────────────────────────────────────────────────────────

x_grid = fm["x"]
y_grid = fm["y"]
z_grid = fm["z"]

# Find the bin index that contains each coordinate
ix = int(np.searchsorted(x_grid, X_VAL, side="right")) - 1
iy = int(np.searchsorted(y_grid, Y_VAL, side="right")) - 1
iz = int(np.searchsorted(z_grid, Z_VAL, side="right")) - 1
ix = np.clip(ix, 0, h.nfx - 1)
iy = np.clip(iy, 0, h.nfy - 1)
iz = np.clip(iz, 0, h.nfz - 1)

print(f"Point ({X_VAL}, {Y_VAL}, {Z_VAL}) falls in:")
print(f"  x bin {ix}: [{x_grid[ix]:.2f}, {x_grid[ix+1]:.2f}]")
print(f"  y bin {iy}: [{y_grid[iy]:.2f}, {y_grid[iy+1]:.2f}]")
print(f"  z bin {iz}: [{z_grid[iz]:.2f}, {z_grid[iz+1]:.2f}]")
print()

print(f"Weight window values at this cell for all energies:")
print(f"{'Bin':>4} {'E range (MeV)':>26} {'WW value':>14}")
print("-" * 48)
for e in range(arr.shape[1]):
    e_lo = 0.0 if e == 0 else e_bins[e - 1]
    e_hi = e_bins[e]
    val = arr[0, e, iz, iy, ix]
    print(f"{e:4d} [{e_lo:10.4e}, {e_hi:10.4e}] {val:14.6e}")

## 5. Query a spatial region

Use the `query()` method to slice a subregion and optionally convert to DataFrame.

In [ ]:
# ── Pick a region and energy ──────────────────────────────────
X_RANGE = (-50, 50)
Y_RANGE = (-50, 50)
Z_RANGE = (-10, 10)
ENERGY_IDX = 0       # which energy bin to display (0-based)
# ──────────────────────────────────────────────────────────────

result = ww.query(particle=0, x=X_RANGE, y=Y_RANGE, z=Z_RANGE)
vals = result.ww_values[0]  # (nt, ne, nfz_sub, nfy_sub, nfx_sub)
print(f"Query shape: {vals.shape}")

# Show the spatial slice for the chosen energy
spatial_slice = vals[0, ENERGY_IDX]  # (nfz_sub, nfy_sub, nfx_sub)
e_lo = 0.0 if ENERGY_IDX == 0 else e_bins[ENERGY_IDX - 1]
e_hi = e_bins[ENERGY_IDX]
print(f"Energy bin {ENERGY_IDX}: [{e_lo:.4e}, {e_hi:.4e}] MeV")
print(f"Spatial slice shape: {spatial_slice.shape}")
print(f"  min={spatial_slice.min():.4e}  max={spatial_slice.max():.4e}  mean={spatial_slice.mean():.4e}")

## 6. 2-D slice plot (z = mid-plane)

Quick visual of weight windows on an XY plane at a fixed z.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ── Parameters ────────────────────────────────────────────────
Z_SLICE = 0.0       # z coordinate for the slice (cm)
E_IDX   = 0          # energy bin index
# ──────────────────────────────────────────────────────────────

iz = int(np.clip(np.searchsorted(fm["z"], Z_SLICE, side="right") - 1, 0, h.nfz - 1))
e_lo = 0.0 if E_IDX == 0 else e_bins[E_IDX - 1]
e_hi = e_bins[E_IDX]

data_2d = arr[0, E_IDX, iz, :, :]  # (nfy, nfx)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.pcolormesh(
    fm["x"], fm["y"], data_2d,
    norm=LogNorm(vmin=data_2d[data_2d > 0].min(), vmax=data_2d.max()),
    cmap="viridis", shading="flat",
)
cb = fig.colorbar(im, ax=ax, label="WW lower bound")
ax.set_xlabel("x (cm)")
ax.set_ylabel("y (cm)")
ax.set_title(f"WW slice at z={fm['z'][iz]:.1f}–{fm['z'][iz+1]:.1f} cm, E=[{e_lo:.2e}, {e_hi:.2e}] MeV")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 7. Compare all energy bins at one point

Shows how the weight window value varies across energies at a fixed spatial location.

In [ ]:
# ── Pick a point ──────────────────────────────────────────────
XP, YP, ZP = 0.0, 0.0, 0.0
# ──────────────────────────────────────────────────────────────

ix = int(np.clip(np.searchsorted(fm["x"], XP, side="right") - 1, 0, h.nfx - 1))
iy = int(np.clip(np.searchsorted(fm["y"], YP, side="right") - 1, 0, h.nfy - 1))
iz = int(np.clip(np.searchsorted(fm["z"], ZP, side="right") - 1, 0, h.nfz - 1))

ww_vs_e = arr[0, :, iz, iy, ix]  # shape (ne,)
e_mids = np.array([(0.0 if i == 0 else e_bins[i-1]) + e_bins[i] for i in range(len(e_bins))]) / 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(e_bins)), ww_vs_e, color="steelblue", edgecolor="k", linewidth=0.5)
ax.set_yscale("log")
ax.set_xlabel("Energy bin index")
ax.set_ylabel("WW lower bound")
ax.set_title(f"WW vs energy at ({XP}, {YP}, {ZP}) cm")

# Add energy labels on x-axis
ax.set_xticks(range(len(e_bins)))
ax.set_xticklabels([f"{e:.2e}" for e in e_bins], rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()